In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
corpus = [
    "machine learning is amazing",
    "deep learning is powerful",
    "transformers are revolutionizing nlp",
    "nlp is a part of artificial intelligence",
    "tensorflow makes deep learning easier"
]

In [3]:
token = Tokenizer()
token.fit_on_texts(corpus)

In [4]:
total_words = len(token.word_index) + 1
total_words

19

# setting up all the thing vocabulary in order

In [5]:
input_sequences = []

for line in corpus:
    token_list = token.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Padding for vocab

In [7]:
max_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# MaskedSelf attention

In [ ]:
class MaskedSelfAttention(tf.keras.layers.Layer):

    def __init__(self, embed_dim):
        super().__init__()

        self.query = tf.keras.layers.Dense(embed_dim)
        self.key = tf.keras.layers.Dense(embed_dim)
        self.value = tf.keras.layers.Dense(embed_dim)

    def call(self, x):

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = tf.matmul(Q, K, transpose_b=True)
        seq_len = tf.shape(scores)[-1]
        mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        scores = scores * mask - 1e9 * (1 - mask)
        attention = tf.nn.softmax(scores)

        return tf.matmul(attention, V)

** Decoder Model

In [ ]:
embed_dim = 32

inputs = tf.keras.layers.Input(shape=(max_len-1,))

x = tf.keras.layers.Embedding(total_words, embed_dim)(inputs)
attention = MaskedSelfAttention(embed_dim)(x)
x = tf.keras.layers.GlobalAveragePooling1D()(attention)

outputs = tf.keras.layers.Dense(total_words, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

In [10]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [11]:
model.fit(X, y, epochs=200)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0526 - loss: 2.9535
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.0526 - loss: 2.9486
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.0526 - loss: 2.9437
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.0526 - loss: 2.9388
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.0526 - loss: 2.9339
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.0526 - loss: 2.9291
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.0526 - loss: 2.9243
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.0526 - loss: 2.9194
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.0526 - loss: 2.9145
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.1579 - loss: 2.9096
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.1579 - loss: 2.9046
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.1579 - loss

In [18]:
seed_text = "machine lernng"

token_list = token.texts_to_sequences([seed_text])[0]
token_list = pad_sequences([token_list], maxlen=max_len-1, padding="pre")

pred = model.predict(token_list)
predicted_word = token.index_word[np.argmax(pred)]

print("Next word:", predicted_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
Next word: is
